# Jamii Afya Falcon deployment baseline

CPU-only, scalar llama.cpp, profiler-shaped `-p 512 -n 128 -ngl 0`. Each quantization gets three repetitions plus deterministic CLI output hashes. This notebook never trains or uses the frozen quality holdout.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
OUT = WORK / 'falcon-deployment-baseline'
BRANCH = 'research/edge35-adaptive-streaming'
OUT.mkdir(parents=True, exist_ok=True)
def run(command, cwd=None, name='command.log'):
    path = OUT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with path.open('a', encoding='utf-8', buffering=1) as log:
        proc = subprocess.Popen([str(x) for x in command], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            rendered = f'[{stamp}] {line}'
            print(rendered, end='', flush=True); log.write(rendered); log.flush()
        code = proc.wait()
        stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
        print('[{}] EXIT={}'.format(stamp, code), flush=True); log.write('[{}] EXIT={}\n'.format(stamp, code)); log.flush()
        if code: raise RuntimeError(f'command failed: {command}')
    return path
if not (REPO / '.git').exists():
    run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
else:
    run(['git','-C',str(REPO),'fetch','origin',BRANCH], cwd=WORK, name='git-refresh.log')
    run(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH], cwd=WORK, name='git-refresh.log')
run([sys.executable,'-m','pip','install','-q','psutil'], cwd=REPO, name='pip.log')
run(['bash','scripts/build_llamacpp_scalar.sh'], cwd=REPO, name='scalar-build.log')
llama = REPO / 'llama.cpp'
cpu_line = next((line.strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')), 'unknown')
print(json.dumps({'repo_sha': subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip(), 'llama_cpp_sha': subprocess.run(['git','rev-parse','HEAD'], cwd=llama, text=True, capture_output=True, check=True).stdout.strip(), 'cpu': cpu_line, 'cores': os.cpu_count()}, indent=2), flush=True)


In [ ]:
models = {
    'deep-Q4_0': 'Falcon-H1-1.5B-Deep-Instruct-Q4_0.gguf',
    'deep-Q4_K_M': 'Falcon-H1-1.5B-Deep-Instruct-Q4_K_M.gguf',
}
MODEL_DIR = WORK / 'falcon-models'; MODEL_DIR.mkdir(exist_ok=True)
base = 'https://huggingface.co/unsloth/Falcon-H1-1.5B-Deep-Instruct-GGUF/resolve/main/'
for tag, filename in models.items():
    destination = MODEL_DIR / filename
    if not destination.exists():
        run(['curl','-L','--fail','--retry','4','-C','-','-o',str(destination),base+filename], name=f'download-{tag}.log')
    print(json.dumps({'tag':tag,'filename':filename,'bytes':destination.stat().st_size}, sort_keys=True), flush=True)


In [ ]:
result_dirs = {}
for tag, filename in models.items():
    result_dir = OUT / tag
    result_dirs[tag] = result_dir
    run([sys.executable,'-u','scripts/benchmark_falcon_deployment.py','--model',str(MODEL_DIR / filename),'--llama-bench',str(llama / 'build-scalar' / 'bin' / 'llama-bench'),'--llama-cli',str(llama / 'build-scalar' / 'bin' / 'llama-cli'),'--out',str(result_dir),'--tag',tag,'--threads','4','--repetitions','3'], cwd=REPO, name=f'benchmark-{tag}.log')
summary = {'schema_version':'1.0.0','experiment_id':'falcon-deployment-baseline','repo_sha':subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,text=True,capture_output=True,check=True).stdout.strip(),'llama_cpp_sha':subprocess.run(['git','rev-parse','HEAD'],cwd=llama,text=True,capture_output=True,check=True).stdout.strip(),'models':{tag:json.loads((path/'deployment_baseline.json').read_text()) for tag,path in result_dirs.items()}}
(OUT/'deployment_baseline_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
for tag, item in summary['models'].items():
    aggregate = item['aggregate']
    print(json.dumps({'tag':tag,'bytes':item['model_bytes'],'sha256':item['model_sha256'],'prompt_tps_mean':aggregate['prompt_tps']['mean'],'decode_tps_mean':aggregate['decode_tps']['mean'],'decode_tps_stdev':aggregate['decode_tps']['stdev'],'peak_rss_mean_mb':aggregate['peak_tree_rss_mb_sampled']['mean'],'steady_rss_mean_mb':aggregate['steady_tree_rss_mb_mean_last_half']['mean'],'deterministic':item['deterministic_generation']['deterministic']}, sort_keys=True), flush=True)
print('DEPLOYMENT_BASELINE_COMPLETE', OUT, flush=True)
